# **Koi**

This model is named after a Georgian journalist, Madona **Koi**dze (მადონა კოიძე), better known as the host of the Georgian investigative TV show **"სახალხო კონტროლი"** (Public Control), where she conducts unannounced inspections of local restaurants, bakeries, markets, and food production facilities. Obviously, this is a reference to the fact that this model is supposed to classify a product into fresh and rotten categories.

This model is a fine-tuned PyTorch vision model built on a pre-trained **EfficientNet-B0** backbone and then fine-tuned on the [Food Freshness Dataset by ULNN Project from Kaggle](https://www.kaggle.com/datasets/ulnnproject/food-freshness-dataset).

I love putting all the imports together, sorted alphabetically, starting with `imports` first, then `from ... import ...`:

In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
import zipfile

from google.colab import drive
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

Mounting drive and loading the dataset:

In [2]:
drive.mount("/content/drive")
zip_path = "/content/drive/MyDrive/Food_Freshness_Dataset.zip"
extract_path = "/content/dataset_extracted"

# unzipping the file into the local runtime environment for faster training disk I/O
if not os.path.exists(extract_path):
    print("Unzipping dataset...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print("Unzipping complete.")

dataset_root = os.path.join(extract_path, "Dataset")
fresh_dir = os.path.join(dataset_root, "Fresh")
rotten_dir = os.path.join(dataset_root, "Rotten")

# skipping hidden operating system files like .DS_Store
fresh_subdirs = sorted([d for d in os.listdir(fresh_dir) if os.path.isdir(os.path.join(fresh_dir, d))])
rotten_subdirs = sorted([d for d in os.listdir(rotten_dir) if os.path.isdir(os.path.join(rotten_dir, d))])

def sanitize_crop_name(name):
    clean = name.replace("Fresh", "").replace("Rotten", "").lower().strip()
    if "cap" in clean:
        return "capsicum"
    if "ok" in clean:
        return "okra"
    return clean

# build dictionaries mapping the sanitized generic name back to its actual folder name on disk
fresh_clean_map = {sanitize_crop_name(d): d for d in fresh_subdirs}
rotten_clean_map = {sanitize_crop_name(d): d for d in rotten_subdirs}

common = sorted(list(set(fresh_clean_map.keys()) & set(rotten_clean_map.keys())))

# trust, but verify
assert len(common) > 0, "Error: No matching common names could be aligned between Fresh and Rotten!"

NUM_CLASSES = len(common)
print(f"Structure verification passed! Cleaned and matched {NUM_CLASSES} identical crop classes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Structure verification passed! Cleaned and matched 13 identical crop classes.


`Koi` definition:

In [3]:
class Koi(nn.Module):
    def __init__(self, num_produce_classes):
        super(Koi, self).__init__()
        W = EfficientNet_B0_Weights.DEFAULT
        NUM_FEATURES = efficientnet_b0(weights=W).classifier[1].in_features

        self.backbone = efficientnet_b0(weights=W)
        self.backbone.classifier = nn.Identity()

        # Head A: Crop Type Categorization
        self.head_produce = nn.Sequential(
            nn.Dropout(p=0.2, inplace=False),
            nn.Linear(NUM_FEATURES, num_produce_classes)
        )

        # Head B: Freshness Assessment
        self.head_state = nn.Sequential(
            nn.Dropout(p=0.2, inplace=False),
            nn.Linear(NUM_FEATURES, 2)
        )

    def forward(self, x):
        features = self.backbone(x)
        produce_logits = self.head_produce(features)
        state_logits = self.head_state(features)
        return produce_logits, state_logits

Dynamically generate the mappings:

In [4]:
PRODUCE_MAP = {name: idx for idx, name in enumerate(common)}
STATE_MAP = {'Fresh': 0, 'Rotten': 1}

for k, v in PRODUCE_MAP.items():
    print(f"{k}: {v}")

apple: 0
banana: 1
bellpepper: 2
bittergroud: 3
capsicum: 4
carrot: 5
cucumber: 6
mango: 7
okra: 8
orange: 9
potato: 10
strawberry: 11
tomato: 12


Custom `Dataset` definition (the kind that PyTorch expects):

In [5]:
class FoodFreshnessDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.produce_labels = []
        self.state_labels = []

        configs = [
            {'name': 'Fresh', 'lookup': fresh_clean_map, 'dir': fresh_dir},
            {'name': 'Rotten', 'lookup': rotten_clean_map, 'dir': rotten_dir}
        ]

        for config in configs:
            s = STATE_MAP[config["name"]]
            d = config["dir"]

            if not os.path.exists(d):
                continue

            for fld in os.listdir(d):
                fld_path = os.path.join(d, fld)
                if not os.path.isdir(fld_path):
                    continue

                sanitized = sanitize_crop_name(fld)
                if sanitized in PRODUCE_MAP:
                    p = PRODUCE_MAP[sanitized]
                    for img_name in os.listdir(fld_path):
                        if img_name.lower().endswith((".png", ".jpg", ".jpeg")):
                            self.image_paths.append(os.path.join(fld_path, img_name))
                            self.state_labels.append(s)
                            self.produce_labels.append(p)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, self.produce_labels[idx], self.state_labels[idx]

Auto-detect TPU, GPU, or CPU environment runtime wrappers and training configuration:

In [6]:
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print("Running optimization processing on Google TPU Matrix.")
except ImportError:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running optimization processing on: {device}")

BS = 32
N_WORKERS = 2
LR = 1e-4
EPOCHS = 3

pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = FoodFreshnessDataset(root_dir=dataset_root, transform=pipeline)
train_loader = DataLoader(train_dataset, batch_size=BS, shuffle=True, num_workers=N_WORKERS)

madona = Koi(num_produce_classes=NUM_CLASSES).to(device)

criterion_produce = nn.CrossEntropyLoss()
criterion_state = nn.CrossEntropyLoss()
optimizer = optim.AdamW(madona.parameters(), lr=LR)

print(f"Dataset successfully mapped! Found {len(train_dataset)} processable images.")

Running optimization processing on: cuda
Dataset successfully mapped! Found 71303 processable images.


Fine-tuning:

In [7]:
warnings.filterwarnings("ignore", category=UserWarning, module="PIL")  # to suppress the PIL transparency palette conversion warning that pops up without this

# compute padding widths dynamically for the perfect console alignment (drives me nuts if it NOT aligned)
epoch_width = len(str(EPOCHS))
batch_width = len(str(len(train_loader)))

print("Koi training has started...")

for epoch in range(EPOCHS):
    madona.train()
    running_loss = 0.0
    correct_produce = 0
    correct_state = 0
    total_samples = 0

    for batch_idx, (images, produce_targets, state_targets) in enumerate(train_loader):
        images = images.to(device)
        produce_targets = produce_targets.to(device)
        state_targets = state_targets.to(device)

        optimizer.zero_grad()
        pred_produce, pred_state = madona(images)

        loss_p = criterion_produce(pred_produce, produce_targets)
        loss_s = criterion_state(pred_state, state_targets)
        total_loss = loss_p + loss_s
        total_loss.backward()

        # TPU execution optimization update check
        try:
            import torch_xla.core.xla_model as xm
            xm.optimizer_step(optimizer)
        except ImportError:
            optimizer.step()

        running_loss += total_loss.item()

        _, predicted_p = torch.max(pred_produce, 1)
        _, predicted_s = torch.max(pred_state, 1)

        total_samples += produce_targets.size(0)
        correct_produce += (predicted_p == produce_targets).sum().item()
        correct_state += (predicted_s == state_targets).sum().item()

        if (batch_idx + 1) % 20 == 0:
            p_epoch = f"{epoch + 1:>{epoch_width}} out of {EPOCHS}"
            p_batch = f"{batch_idx + 1:>{batch_width}}/{len(train_loader)}"
            print(f"Epoch {p_epoch} | Batch: {p_batch} | Loss: {total_loss.item():.4f}")

    epoch_loss = running_loss / len(train_loader)
    acc_produce = (correct_produce / total_samples) * 100
    acc_state = (correct_state / total_samples) * 100

    print(f"\nEpoch [{epoch + 1}/{EPOCHS}] Metrics Dashboard Summary:")
    print(f" Average System Loss: {epoch_loss:.4f}")
    print(f" Crop Type Accuracy: {acc_produce:.2f}%")
    print(f" Freshness Accuracy: {acc_state:.2f}%")
    print("-" * 50)


torch.save(madona.state_dict(), "/content/koi.pt")
print("\nKoi Training completed!")

Koi training has started...
Epoch 1 out of 3 | Batch:   20/2229 | Loss: 2.6109
Epoch 1 out of 3 | Batch:   40/2229 | Loss: 1.9350
Epoch 1 out of 3 | Batch:   60/2229 | Loss: 1.2324
Epoch 1 out of 3 | Batch:   80/2229 | Loss: 1.2218
Epoch 1 out of 3 | Batch:  100/2229 | Loss: 1.1481
Epoch 1 out of 3 | Batch:  120/2229 | Loss: 0.8874
Epoch 1 out of 3 | Batch:  140/2229 | Loss: 0.6904
Epoch 1 out of 3 | Batch:  160/2229 | Loss: 0.5296
Epoch 1 out of 3 | Batch:  180/2229 | Loss: 0.4847
Epoch 1 out of 3 | Batch:  200/2229 | Loss: 0.6735
Epoch 1 out of 3 | Batch:  220/2229 | Loss: 0.5518
Epoch 1 out of 3 | Batch:  240/2229 | Loss: 0.2521
Epoch 1 out of 3 | Batch:  260/2229 | Loss: 0.4443
Epoch 1 out of 3 | Batch:  280/2229 | Loss: 0.4190
Epoch 1 out of 3 | Batch:  300/2229 | Loss: 0.2982
Epoch 1 out of 3 | Batch:  320/2229 | Loss: 0.3383
Epoch 1 out of 3 | Batch:  340/2229 | Loss: 0.2397
Epoch 1 out of 3 | Batch:  360/2229 | Loss: 0.2587
Epoch 1 out of 3 | Batch:  380/2229 | Loss: 0.1351
Epo